# Heatmap

## Import

In [ ]:
# nur nutzen, wenn man Änderungen in Visualisierungen gemacht hat, aber Kernel nicht neu starten will.
import importlib
import visualisierungen
import re

importlib.reload(visualisierungen)

In [ ]:
from visualisierungen import heatmap, heatmap_interaktiv_phasen
import pandas as pd


PARTEI_EXCLUDE = {"p-glp_label"}

PARTEI_ORDER = [
    "br-pos_label",
    "bv-pos_label",
    "p-gps_label",
    "p-sps_label",
    "p-mitte_label",
    "p-fdp_label",
    "p-svp_label",
]


def prepare_heatmap_df(df):
    """partei als Index, ohne GLP, feste Zeilenreihenfolge."""
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    if "partei" in df.columns:
        df = df[~df["partei"].isin(PARTEI_EXCLUDE)].set_index("partei")
    else:
        df = df[~df.index.isin(PARTEI_EXCLUDE)]
    order = [p for p in PARTEI_ORDER if p in df.index]
    return df.loc[order]


def to_kongruenz_scale(M):
    """CSV aus 2_berechnung: (ja_proz − 50) in Prozentpunkten → Plot −0.5 … +0.5."""
    M = M.drop(columns=["jahr", "phase", "abstimmung"], errors="ignore").astype(float) #Fallback Sicherheit
    if M.abs().max().max() > 1: # weil df, erstes max für column und zweites max innerhalb der column
        M = M / 100
    return M.clip(-0.5, 0.5)


def party_short(name):
    s = str(name)
    if "-" not in s or "_" not in s:
        return s
    pre, rest = s.split("-", 1)
    mid = rest.split("_", 1)[0]
    if mid == "pos":
        if pre == "br":
            return "Bundesrat"
        if pre == "bv":
            return "Bundesver."
    if pre == "p":
        return {"gps": "Grüne", "sps": "SP", "mitte": "Mitte", "fdp": "FDP", "svp": "SVP"}.get(mid, mid.upper())
    return f"{pre.upper()}-{mid.upper()}"

## Heatmap all

In [ ]:
df = pd.read_csv("../data/processed/df_heatmap_with_positions.csv")
df.head()

In [ ]:
df_plot = to_kongruenz_scale(prepare_heatmap_df(df))
df_plot

In [ ]:
cantons = [str(c)[:2].upper() for c in df_plot.columns] # Macht die ersten zwei Ziffern mit Grossbuchstaben. Bsp. zh = ZH

parties = [party_short(i) for i in df_plot.index]

heatmap(
    df_plot,
    xlabel="Kantone",
    ylabel="Akteure",
    xlabels=cantons,
    ylabels=parties,
    figsize=(16, 6),
)

## Heatmap timeslot

In [ ]:
df_phase = pd.read_csv("../data/processed/df_heatmap_by_phase.csv")
if "Unnamed: 0" in df_phase.columns:
    df_phase = df_phase.drop(columns=["Unnamed: 0"]) # Fallback, wenn CSV Index-Spalte enthält, diese aber keinen Header hat.
df_phase = df_phase[~df_phase["partei"].isin(PARTEI_EXCLUDE)]
df_phase.head()

In [ ]:
# Slugs kommen aus df_phase; Titel hier (an pd.cut in 2_berechnung anpassen)
PHASE_TITLES = {
    "phase1_fruehphase": "Frühphase (1848–1899)",
    "phase2_volatile": "Volatile Phase (1900–1949)",
    "phase3_konsens": "Konsensphase (1950–1975)",
    "phase4_aufspaltung": "Aufspaltung (1976–2009)",
    "phase5_2010_heute": "2010er–heute",
}


def _phase_order(slug: str) -> int:
    m = re.search(r"phase(\d+)", str(slug))
    return int(m.group(1)) if m else 99


phase_slugs = sorted(df_phase["phase"].dropna().unique(), key=_phase_order)
PHASES = [(slug, PHASE_TITLES.get(slug, slug)) for slug in phase_slugs]

In [ ]:
phasen_plot = []
for slug, titel in PHASES:
    sub = df_phase.loc[df_phase["phase"] == slug]
    if sub.empty:
        print(f"Übersprungen (keine Daten): {titel}")
        continue
    df_plot = to_kongruenz_scale(
        prepare_heatmap_df(sub.drop(columns=["phase"], errors="ignore"))
    )
    phasen_plot.append((titel, df_plot))

if phasen_plot:
    cantons = [str(c)[:2].upper() for c in phasen_plot[0][1].columns]
    parties = [party_short(i) for i in phasen_plot[0][1].index]

    fig_phase = heatmap_interaktiv_phasen(
        phasen_plot,
        xlabel="Kantone",
        ylabel="Akteure",
        xlabels=cantons,
        ylabels=parties,
        height=420,
    )
    fig_phase.show()
    fig_phase.write_html(
        "../Blog/blog_plots/d6_heatmap_zeitphasen.html",
        include_plotlyjs="inline",
        full_html=True,
    )
else:
    print("Keine Phasen-Daten für Heatmap.")